# 01. Data Intake and Audit

This notebook creates a transparent record of the raw dataset before any modeling occurs. The audit establishes whether the input contains the expected columns, whether controls exist for all cell types, and whether dose and perturbation metadata are sufficiently complete for residual perturbation-response analysis. Because downstream claims depend on the validity of these metadata fields, the audit is treated as a first-class methodological step.


In [2]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM_repo
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [5]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [6]:
import hashlib
import anndata as ad

DATA_PATH = DEFAULT_H5AD
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"

adata = ad.read_h5ad(DATA_PATH)
print(adata)
print("obs columns:", list(adata.obs.columns))

if "dose_val" in adata.obs.columns and "dose" not in adata.obs.columns:
    adata.obs["dose"] = adata.obs["dose_val"]

required_cols = ["condition", "cell_type", "dose"]
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

raw_fingerprint = {
    "data_path": str(DATA_PATH),
    "n_obs": int(adata.n_obs),
    "n_vars": int(adata.n_vars),
    "obs_columns": list(adata.obs.columns),
    "file_size_bytes": int(DATA_PATH.stat().st_size),
}
save_json(raw_fingerprint, INTERIM_DIR / "raw_data_fingerprint.json")
raw_fingerprint


AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
obs columns: ['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sco

{'data_path': '/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad',
 'n_obs': 354640,
 'n_vars': 2000,
 'obs_columns': ['cell_type',
  'dose',
  'dose_character',
  'dose_pattern',
  'g1s_score',
  'g2m_score',
  'pathway',
  'pathway_level_1',
  'pathway_level_2',
  'product_dose',
  'product_name',
  'proliferation_index',
  'replicate',
  'size_factor',
  'target',
  'vehicle',
  'batch',
  'n_counts',
  'dose_val',
  'condition',
  'drug_dose_name',
  'cov_drug_dose_name',
  'cov_drug',
  'control',
  'split_ho_pathway',
  'split_tyrosine_ood',
  'split_epigenetic_ood',
  'split_cellcycle_ood',
  'SMILES',
  'split_ood_finetuning',
  'split_ho_epigenetic',
  'split_ho_epigenetic_all',
  'split_random'],
 'file_size_bytes': 417009839}

## Schema and missingness audit

The next cells summarize the observation schema, quantify missingness, and establish the coverage of cell types, conditions, and doses. These summaries become the reference point for all later split and evaluation decisions.


In [7]:
obs_schema = pd.DataFrame({
    "column": adata.obs.columns,
    "dtype": [str(adata.obs[c].dtype) for c in adata.obs.columns],
    "n_missing": [int(adata.obs[c].isna().sum()) for c in adata.obs.columns],
    "n_unique": [int(adata.obs[c].nunique(dropna=True)) for c in adata.obs.columns],
})
obs_schema.to_csv(INTERIM_DIR / "obs_schema.csv", index=False)

missingness = pd.DataFrame({
    "column": adata.obs.columns,
    "missing_fraction": [float(adata.obs[c].isna().mean()) for c in adata.obs.columns],
}).sort_values("missing_fraction", ascending=False)
missingness.to_csv(INTERIM_DIR / "missingness_report.csv", index=False)

cell_counts = adata.obs["cell_type"].astype(str).value_counts().rename_axis("cell_type").reset_index(name="count")
condition_counts = adata.obs["condition"].astype(str).value_counts().rename_axis("condition").reset_index(name="count")
dose_summary = adata.obs.groupby("condition", dropna=False)["dose"].agg(["count", "min", "median", "max", "nunique"]).reset_index()

cell_counts.to_csv(INTERIM_DIR / "cell_type_counts.csv", index=False)
condition_counts.to_csv(INTERIM_DIR / "condition_counts.csv", index=False)
dose_summary.to_csv(INTERIM_DIR / "dose_summary.csv", index=False)

summary = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_conditions": int(adata.obs["condition"].astype(str).nunique()),
    "n_cell_types": int(adata.obs["cell_type"].astype(str).nunique()),
    "n_controls": int((adata.obs["condition"].astype(str) == "control").sum()),
}
pd.DataFrame([summary]).to_csv(INTERIM_DIR / "dataset_summary.csv", index=False)
pd.DataFrame([summary])


/tmp/ipykernel_8399/2348248627.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dose_summary = adata.obs.groupby("condition", dropna=False)["dose"].agg(["count", "min", "median", "max", "nunique"]).reset_index()


,n_cells,n_genes,n_conditions,n_cell_types,n_controls
0,354640,2000,188,3,13004


In [8]:
red_flags = []

control_mask = adata.obs["condition"].astype(str) == "control"
for cell in sorted(adata.obs["cell_type"].astype(str).unique()):
    n_ctrl = int((control_mask & (adata.obs["cell_type"].astype(str) == cell)).sum())
    if n_ctrl == 0:
        red_flags.append({"issue": "missing_control_for_cell_type", "cell_type": cell})

neg_dose = adata.obs["dose"].fillna(0).astype(float) < 0
if int(neg_dose.sum()) > 0:
    red_flags.append({"issue": "negative_dose_rows", "count": int(neg_dose.sum())})

single_dose_conditions = (
    adata.obs.groupby("condition")["dose"].nunique().reset_index().query("dose <= 1")
)
for _, row in single_dose_conditions.iterrows():
    red_flags.append({"issue": "condition_has_single_unique_dose", "condition": row["condition"], "n_unique_dose": int(row["dose"])})

red_flags_df = pd.DataFrame(red_flags)
red_flags_df.to_csv(INTERIM_DIR / "audit_red_flags.csv", index=False)
red_flags_df.head(20)


/tmp/ipykernel_8399/1017911292.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby("condition")["dose"].nunique().reset_index().query("dose <= 1")


,issue,condition,n_unique_dose
0,condition_has_single_unique_dose,control,1


The audit outputs should be reviewed before modeling begins. In particular, missing controls, negative doses, or conditions with only one observed dose can alter the interpretation of residual targets and dose-response diagnostics. The next notebook formalizes split construction and checks that train, test, and OOD assignments are leakage-free.
